# 01 · Feature profile

Distribution of every published feature at a chosen geographic level: coverage, central tendency,
dispersion, skew and spatial pattern. Run `cf run` first to populate `data/marts/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from city_fabric import analysis as cfa

sns.set_theme(style="whitegrid", context="notebook", font_scale=0.9)
pd.set_option("display.max_columns", 50, "display.width", 200, "display.float_format", "{:,.4g}".format)

In [ ]:
LEVEL = "nta"  # borough | community_district | nta | census_tract | zcta
df = cfa.load_features(LEVEL)
cat = cfa.catalog()
print(f"{LEVEL}: {len(df)} geographies × {len(cat)} features")
cfa.manifest()["built_at"]

## Feature catalog

In [ ]:
cat.groupby(["dataset", "kind"]).size().unstack(fill_value=0)

## Summary statistics
High `skew` or `cv` suggests a log transform before correlating; high `zeros` flags sparse features.

In [ ]:
prof = cfa.profile(df)
prof.sort_values("skew", ascending=False)

## Distributions (density features)

In [ ]:
feats = cfa.feature_ids(kind="per_km2")[:16]
ncol = 4
fig, axes = plt.subplots(int(np.ceil(len(feats) / ncol)), ncol, figsize=(14, 2.6 * np.ceil(len(feats) / ncol)))
for ax, f in zip(axes.flat, feats):
    sns.histplot(df[f].dropna(), bins=40, ax=ax, color="#3b6fb6", edgecolor=None)
    ax.set_title(cat.loc[f, "title"], fontsize=8, loc="left")
    ax.set_xlabel(""); ax.set_ylabel("")
for ax in axes.flat[len(feats):]:
    ax.set_visible(False)
fig.tight_layout()

## Spatial pattern

In [ ]:
MAP_FEATURES = ["trees_live_per_km2", "trees_planted_5y_per_km2", "sr311_total_per_km2", "sr311_median_days_to_close"]
MAP_FEATURES = [f for f in MAP_FEATURES if f in df.columns]
fig, axes = plt.subplots(1, len(MAP_FEATURES), figsize=(4.5 * len(MAP_FEATURES), 5))
for ax, f in zip(np.atleast_1d(axes), MAP_FEATURES):
    cfa.choropleth(LEVEL, df[f], ax=ax, title=cat.loc[f, "title"])
fig.tight_layout()

## Borough breakdown

In [ ]:
if df["borough"].notna().any():
    display(df.groupby("borough")[MAP_FEATURES].median())

## 311 complaint mix
Share of each top complaint type, by borough (median across geographies).

In [ ]:
share_cols = cfa.feature_ids(kind="share", dataset="service_requests_311")
if share_cols and df["borough"].notna().any():
    mix = df.groupby("borough")[share_cols].median().T
    mix.index = cat.loc[mix.index, "title"].str.replace(r"^311: | \(share\)$", "", regex=True)
    fig, ax = plt.subplots(figsize=(8, 9))
    sns.heatmap(mix, cmap="mako_r", annot=True, fmt=".1%", cbar=False, ax=ax, annot_kws={"size": 7})
    ax.set_xlabel(""); ax.set_ylabel("")